In [2]:
from datasets import load_dataset

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading IMDB sentiment dataset 

In [3]:
dataset = load_dataset("imdb")
print(dataset)

Generating unsupervised split: 100%|██████████████████████████| 50000/50000 [00:00<00:00, 198426.70 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
# Exploring dataset 
print("\nExample review:")
print(dataset['train'][0]['text'][:200])
print("\nLabel:", dataset['train'][0]['label'])
print("(0 = negative, 1 = positive)")



Example review:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev

Label: 0
(0 = negative, 1 = positive)


In [5]:
# Check class balance
train_labels = dataset['train']['label']
print(f"\nPositive reviews: {sum(train_labels)}")
print(f"Negative reviews: {len(train_labels) - sum(train_labels)}")


Positive reviews: 12500
Negative reviews: 12500


# Loading pretrained model

In [6]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch 
from torch.utils.data import Dataset, DataLoader

In [7]:
# Cutting on dataset size to speed up the training 
train_data = dataset['train'].select(range(1000))
test_data = dataset['test'].select(range(200))

In [8]:
# LOading tokenizer 
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [9]:
# Custome Dataset Class
class IMDbDataset(Dataset):
    def __init__(self,data,tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        text = self.data[idx]['text']
        label = self.data[idx]['label']

        # TOkenize - truncate to maxlength
        encoding = self.tokenizer(
            text,
            max_length= self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return{
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask':encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label,dtype = torch.long)
        }

In [10]:
# Create datasets and dataloaders
train_dataset = IMDbDataset(train_data, tokenizer)
test_dataset = IMDbDataset(test_data, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [11]:
print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 63
Test batches: 13


# BertForSequenceClassification

In [12]:
# BertForSequenceClassification = BERT + linear classification head on [CLS] token
# num_labels=2 — positive or negative
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights: 100%|█████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3879.43it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized becau

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
# fine tuune everythin with a small lr because using big learning rate will cause catastrophic forgetting 

In [18]:
# Training loop
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['label']
        )

        loss = outputs.loss  # BertForSequenceClassification computes loss internally
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


In [19]:
# Evaluation
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask']
            )
            predictions = torch.argmax(outputs.logits, dim=1)
            correct += (predictions == batch['label']).sum().item()
            total += len(batch['label'])

    return correct / total

In [20]:
# Train for 3 epochs
for epoch in range(3):
    avg_loss = train_epoch(model, train_loader, optimizer)
    accuracy = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.4f}")
    

Epoch 1, Loss: 0.0967, Test Accuracy: 1.0000
Epoch 2, Loss: 0.0027, Test Accuracy: 1.0000
Epoch 3, Loss: 0.0014, Test Accuracy: 1.0000
